# 2일차 실습 2. 텍스트 처리 API 구현

## 실습 목표

이번 실습에서는 FastAPI와 Pydantic 모델을 사용해 텍스트 처리 API를 구현합니다.

구현할 API는 다음과 같습니다.

```text
POST /text/process
```

입력 예시:

```json
{
  "text": "hello fastapi",
  "mode": "upper"
}
```

응답 예시:

```json
{
  "result": "HELLO FASTAPI"
}
```

이번 실습에서는 실제 LLM API를 호출하지 않고, 규칙 기반 텍스트 처리 로직으로 API 흐름을 구현합니다.

## 1. 라이브러리 불러오기

FastAPI 앱 생성, 요청 테스트, 데이터 모델 정의에 필요한 라이브러리를 불러옵니다.

In [2]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
import json

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 구현할 API 구조 살펴보기

이번 실습에서 만들 기본 API는 입력 텍스트와 처리 방식을 받아 결과를 반환합니다.

```text
POST /text/process
```

지원할 기본 `mode`는 다음과 같습니다.

```text
upper   → 대문자로 변환
lower   → 소문자로 변환
reverse → 문자열 뒤집기
length  → 글자 수 반환
```

## 3. FastAPI 앱 만들기

텍스트 처리 API를 등록할 FastAPI 앱 객체를 생성합니다.

In [3]:
app = FastAPI(title="Text Processing API")

## 4. Request Model 정의하기

Request Model은 클라이언트가 보내는 입력 Body의 구조를 정의합니다.

이번 API는 `text`와 `mode`를 입력으로 받습니다.

In [4]:
class TextProcessRequest(BaseModel):
    text: str
    mode: str = "upper"

## 5. Response Model 정의하기

Response Model은 서버가 반환하는 응답 Body의 구조를 정의합니다.

기본 API는 처리 결과를 `result` 필드로 반환합니다.

In [5]:
class TextProcessResponse(BaseModel):
    result: str

## 6. 텍스트 처리 함수 만들기

Endpoint 안에 모든 로직을 직접 작성하지 않고, 텍스트 처리 함수를 따로 만듭니다.

이렇게 하면 요청을 받는 부분과 실제 처리 로직을 분리해서 볼 수 있습니다.

In [6]:
def process_text(text: str, mode: str) -> str:
    if mode == "upper":
        return text.upper()

    if mode == "lower":
        return text.lower()

    if mode == "reverse":
        return text[::-1]

    if mode == "length":
        return str(len(text))

    return text

## 7. 처리 함수 실행해보기

서버 Endpoint에 연결하기 전에, 처리 함수가 원하는 결과를 반환하는지 먼저 실행해봅니다.

In [7]:
sample_text = "Hello FastAPI"

for mode in ["upper", "lower", "reverse", "length"]:
    result = process_text(sample_text, mode)

    print("=" * 50)
    print("mode:", mode)
    print("result:", result)

mode: upper
result: HELLO FASTAPI
mode: lower
result: hello fastapi
mode: reverse
result: IPAtsaF olleH
mode: length
result: 13


## 8. Endpoint 구현하기

Request Model로 입력을 받고, 텍스트 처리 함수를 호출한 뒤, Response Model 형태로 응답합니다.

In [44]:
@app.post("/text/process", response_model=TextProcessResponse)
def process_text_api(request: TextProcessRequest):
    result = process_text(request.text, request.mode)

    return {"result": result}

## 9. TestClient로 API 요청 보내기

`TestClient`를 사용해 노트북 안에서 API 요청을 실행합니다.

In [41]:
client = TestClient(app)

## 10. 응답 출력 함수 만들기

요청 결과를 보기 좋게 출력하는 함수를 만듭니다.

(실습1에서 만든 `print_api_response` 함수와 동일합니다. 실습1·2가 각각 독립적으로 실행될 수 있도록 다시 정의합니다.)

In [21]:
def print_api_response(response):
    print("=" * 60)
    print("Status Code:", response.status_code)
    print("-" * 60)

    try:
        print("Response Body:")
        print(json.dumps(response.json(), ensure_ascii=False, indent=2))
    except Exception:
        print("Response Text:")
        print(response.text)

    print("=" * 60)

## 11. 기본 mode 요청 테스트하기

`upper`, `lower`, `reverse`, `length` 요청을 차례대로 보내봅니다.

In [27]:
test_requests = [
    {"text": "Hello FastAPI", "mode": "upper"},
    {"text": "Hello FastAPI", "mode": "lower"},
    {"text": "Hello FastAPI", "mode": "reverse"},
    {"text": "Hello FastAPI", "mode": "length"},
]

for request_body in test_requests:
    print("요청 Body:", request_body)
    response = client.post("/text/process", json=request_body)
    print_api_response(response)

요청 Body: {'text': 'Hello FastAPI', 'mode': 'upper'}
Status Code: 200
------------------------------------------------------------
Response Body:
{
  "result": "HELLO FASTAPI"
}
요청 Body: {'text': 'Hello FastAPI', 'mode': 'lower'}
Status Code: 200
------------------------------------------------------------
Response Body:
{
  "result": "hello fastapi"
}
요청 Body: {'text': 'Hello FastAPI', 'mode': 'reverse'}
Status Code: 200
------------------------------------------------------------
Response Body:
{
  "result": "IPAtsaF olleH"
}
요청 Body: {'text': 'Hello FastAPI', 'mode': 'length'}
Status Code: 200
------------------------------------------------------------
Response Body:
{
  "result": "13"
}


## 12. mode 기본값 테스트하기

`mode`를 생략하면 Request Model에 정의한 기본값 `"upper"`가 사용됩니다.

In [12]:
response = client.post("/text/process", json={
    "text": "default mode test"
})

print_api_response(response)

Status Code: 200
------------------------------------------------------------
Response Body:
{
  "result": "DEFAULT MODE TEST"
}


## 13. 서버 파일로 저장하기

아래 셀을 실행하면 지금까지 만든 API 코드를 `main.py` 파일로 저장합니다.

In [13]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Text Processing API")


class TextProcessRequest(BaseModel):
    text: str
    mode: str = "upper"


class TextProcessResponse(BaseModel):
    result: str


def process_text(text: str, mode: str) -> str:
    if mode == "upper":
        return text.upper()

    if mode == "lower":
        return text.lower()

    if mode == "reverse":
        return text[::-1]

    if mode == "length":
        return str(len(text))

    return text


@app.post("/text/process", response_model=TextProcessResponse)
def process_text_api(request: TextProcessRequest):
    result = process_text(request.text, request.mode)

    return {
        "result": result
    }

Overwriting main.py


## 14. FastAPI 자동 문서 화면에서 테스트하기

터미널에서 서버를 실행합니다.

```bash
uvicorn main:app --reload
```

브라우저에서 다음 주소로 접속합니다.

```text
http://127.0.0.1:8000/docs
```

`POST /text/process`를 열고 다양한 `mode` 값을 입력해 실행해봅니다.

# TODO 실습

이제 기본 API를 확장해 `POST /text/process-v2`를 구현합니다.

v2 API에서는 다음 내용을 직접 완성합니다.

- 더 많은 텍스트 처리 모드를 지원하는 함수
- 처리 결과와 메타데이터를 포함하는 응답 모델
- 지원하지 않는 mode에 대한 400 오류 처리
- 정상 요청과 실패 요청 테스트

## TODO 1. `process_text_v2()` 함수 구현하기

`mode` 값에 따라 텍스트를 처리하는 `process_text_v2()` 함수를 구현하세요.

지원해야 하는 mode는 다음과 같습니다.

```text
upper      → 대문자로 변환
lower      → 소문자로 변환
reverse    → 문자열 뒤집기
length     → 글자 수 반환
title      → 단어 첫 글자 대문자 변환
word_count → 단어 수 반환
```

`length`, `word_count` 결과도 문자열로 반환하세요.

In [50]:
def process_text_v2(text: str, mode: str) -> str:
    # TODO 1:
    # mode 값에 따라 텍스트를 처리하세요.
    # 지원 mode: upper, lower, reverse, length, title, word_count
    if mode == "upper":
        return text.upper()
    if mode == "lower":
        return text.lower()
    if mode == "reverse":
        return text[::-1]
    if mode == "length":
        return str(len(text))
    if mode == "title":
        return text.title()
    if mode == "word_count":
        return str(len(text.split()))

    raise HTTPException(status_code=400, detail="지원하지 않는 모드입니다")


test_cases = [
    {"text": "hello fastapi", "mode": "upper"},
    {"text": "Hello FastAPI", "mode": "lower"},
    {"text": "hello", "mode": "reverse"},
    {"text": "hello", "mode": "length"},
    {"text": "hello fastapi", "mode": "title"},
    {"text": "hello fastapi api", "mode": "word_count"},
]

for case in test_cases:
    result = process_text_v2(case["text"], case["mode"])

    print("=" * 50)
    print("입력:", case)
    print("결과:", result)

입력: {'text': 'hello fastapi', 'mode': 'upper'}
결과: HELLO FASTAPI
입력: {'text': 'Hello FastAPI', 'mode': 'lower'}
결과: hello fastapi
입력: {'text': 'hello', 'mode': 'reverse'}
결과: olleh
입력: {'text': 'hello', 'mode': 'length'}
결과: 5
입력: {'text': 'hello fastapi', 'mode': 'title'}
결과: Hello Fastapi
입력: {'text': 'hello fastapi api', 'mode': 'word_count'}
결과: 3


## TODO 2. `TextProcessResponseV2` 모델 정의하기

v2 API의 응답 모델을 정의하세요.

응답에는 다음 필드를 포함합니다.

```text
result: 처리 결과
mode: 사용한 처리 방식
original_length: 원본 텍스트 길이
processed_length: 처리 결과 길이
```

In [51]:
class TextProcessResponseV2(BaseModel):
    # TODO 2:
    # result, mode, original_length, processed_length 필드를 정의하세요.
    result: str
    mode: str
    original_length: int
    processed_length: int

## TODO 3. `/text/process-v2` Endpoint 구현하기

`POST /text/process-v2` Endpoint를 구현하세요.

요구사항은 다음과 같습니다.

1. `TextProcessRequest`로 요청 Body를 받습니다.
2. 지원하지 않는 mode가 들어오면 400 오류를 반환합니다.
3. `process_text_v2()`를 호출해 처리 결과를 만듭니다.
4. `TextProcessResponseV2` 구조에 맞게 응답을 반환합니다.

In [52]:
SUPPORTED_MODES_V2 = {
    "upper",
    "lower",
    "reverse",
    "length",
    "title",
    "word_count",
}


@app.post("/text/process-v2", response_model=TextProcessResponseV2)
def process_text_api_v2(request: TextProcessRequest):
    # TODO 3-1:
    # 지원하지 않는 mode인지 검사하세요.
    if request.mode not in SUPPORTED_MODES_V2:
        raise HTTPException(status_code=400, detail="지원하지 않는 모드입니다")

    # TODO 3-2:
    # process_text_v2()를 호출해 처리 결과를 만드세요.
    result = process_text_v2(request.text, request.mode)

    # TODO 3-3:
    # TextProcessResponseV2 구조에 맞게 응답을 반환하세요.
    return {
        "result": result,
        "mode": request.mode,
        "original_length": len(request.text),
        "processed_length": len(result),
    }

## TODO 4. TestClient로 v2 API 테스트하기

`TestClient`를 사용해 `/text/process-v2`를 테스트하세요.

테스트할 요청은 다음과 같습니다.

```text
정상 요청 1: title mode
정상 요청 2: word_count mode
실패 요청: unknown mode
```

In [53]:
test_requests_v2 = [
    {
        "text": "hello fastapi",
        "mode": "title"
    },
    {
        "text": "hello fastapi api",
        "mode": "word_count"
    },
    {
        "text": "hello fastapi",
        "mode": "unknown"
    },
]

for request_body in test_requests_v2:
    # TODO 4:
    # POST /text/process-v2 요청을 보내고 결과를 출력하세요.
    http_response = client.post("/text/process-v2", json=request_body)
    print_api_response(http_response)


ResponseValidationError: 1 validation error:
  {'type': 'model_attributes_type', 'loc': ('response',), 'msg': 'Input should be a valid dictionary or object to extract fields from', 'input': None}

  File "/var/folders/m9/22pbfsqj2k52h8fn3krgqw740000gn/T/ipykernel_42084/3967156411.py", line 11, in process_text_api_v2
    POST /text/process-v2

## TODO 5. 최종 서버 파일 저장하기

완성한 v2 API를 포함해 `main.py` 파일을 다시 저장하세요.

아래 셀의 TODO 부분을 완성한 뒤 실행합니다.

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Text Processing API")


class TextProcessRequest(BaseModel):
    text: str
    mode: str = "upper"


class TextProcessResponse(BaseModel):
    result: str


class TextProcessResponseV2(BaseModel):
    # TODO 5-1:
    # result, mode, original_length, processed_length 필드를 정의하세요.
    result: str
    mode: str
    original_length: int
    processed_length: int


def process_text(text: str, mode: str) -> str:
    if mode == "upper":
        return text.upper()

    if mode == "lower":
        return text.lower()

    if mode == "reverse":
        return text[::-1]

    if mode == "length":
        return str(len(text))

    return text


def process_text_v2(text: str, mode: str) -> str:
    # TODO 5-2:
    # mode 값에 따라 텍스트를 처리하세요.
    # 지원 mode: upper, lower, reverse, length, title, word_count
    if mode == "upper":
        return text.upper()

    if mode == "lower":
        return text.lower()

    if mode == "reverse":
        return text[::-1]

    if mode == "length":
        return str(len(text))

    if mode == "title":
        return text.title()

    if mode == "word_count":
        return str(len(text.split()))

    raise HTTPException(status_code=400, detail="지원하지 않는 모드입니다")


@app.post("/text/process", response_model=TextProcessResponse)
def process_text_api(request: TextProcessRequest):
    result = process_text(request.text, request.mode)

    return {
        "result": result
    }


SUPPORTED_MODES_V2 = {
    "upper",
    "lower",
    "reverse",
    "length",
    "title",
    "word_count",
}


@app.post("/text/process-v2", response_model=TextProcessResponseV2)
def process_text_api_v2(request: TextProcessRequest):
    # TODO 5-3:
    # 지원하지 않는 mode인지 검사하세요.
    if request.mode not in SUPPORTED_MODES_V2:
        raise HTTPException(status_code=400, detail="지원하지 않는 모드입니다")

    # TODO 5-4:
    # process_text_v2()를 호출해 처리 결과를 만드세요.
    result = process_text_v2(request.text, request.mode)

    # TODO 5-5:
    # TextProcessResponseV2 구조에 맞게 응답을 반환하세요.
    return TextProcessResponseV2(
        result=result,
        mode=request.mode,
        original_length=len(request.text),
        processed_length=len(result)
    )

Overwriting main.py


## 실습 정리

이번 실습에서 구현한 내용은 다음과 같습니다.

- Request Model로 입력 Body 구조를 정의했습니다.
- Response Model로 응답 Body 구조를 정의했습니다.
- 텍스트 처리 로직을 별도 함수로 분리했습니다.
- Endpoint에서 Request Model을 받아 처리 함수를 호출했습니다.
- TestClient와 FastAPI 자동 문서 화면으로 API 요청을 테스트했습니다.
- v2 API에서 응답 구조 확장과 오류 처리를 구현했습니다.